# Home Assignment: Dimensionality Reduction in Chemistry (PCA, t-SNE, UMAP)

**Course:** Machine Learning in Chemistry

**Instructions**
1. In Google Colab, go to *File → Save a copy in Drive* before you start, so your work is saved.
2. Work through the notebook from top to bottom.
3. Write answers in the **Your answer** cells (double-click to edit). Mathematical expressions can be typed with LaTeX, for example `$\lambda_1 = 4$`.
4. Code cells marked *Scratch* are for your own calculations. You may add more cells if needed.
5. When finished, go to *File → Download → Download .ipynb* and submit the notebook.

**Contents**
- Part A: Problems by hand (use the scratch cells to check your arithmetic)
- Part B: Computational practice
- Part C: Short report
- Bonus

## Setup
Run this cell first. It installs RDKit and UMAP (takes about a minute on Colab).

In [ ]:
!pip install -q rdkit umap-learn

---
# Part A: Problems by hand

Show all steps. Round final answers to three significant figures unless stated otherwise. You may use the scratch cells with NumPy to *check* your work, but your answer cells must show the steps.

In [ ]:
import numpy as np

## A1. PCA on two descriptors

Four molecules are described by two descriptors measured on the same scale, $(x_1, x_2)$:

$P = (7, 3), \quad Q = (1, 1), \quad R = (5, 3), \quad S = (3, 1)$

**(a)** Center the data and compute the covariance matrix using $1/N$ normalization.

**(b)** Find both eigenvalues and the corresponding unit eigenvectors. Verify that the eigenvalues sum to the trace of the covariance matrix.

**(c)** What fraction of the total variance is explained by PC1? What angle does PC1 make with the $x_1$ axis?

**(d)** Compute the PC1 scores of all four molecules. Show that the mean squared reconstruction error, when only PC1 is kept, equals the discarded eigenvalue.

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
# Scratch: check A1 here
X = np.array([[7, 3], [1, 1], [5, 3], [3, 1]], dtype=float)

## A2. t-SNE conditional probabilities

A molecule $i$ has three neighbors at distances 1, 1, and 2 in descriptor space. Use

$$p_{j|i} = \frac{\exp(-d_{ij}^2 / 2\sigma_i^2)}{\sum_k \exp(-d_{ik}^2 / 2\sigma_i^2)}$$

**(a)** Compute all three $p_{j|i}$ for $\sigma_i = 1$.

**(b)** Compute the entropy $H = -\sum_j p_{j|i} \log_2 p_{j|i}$ (in bits) and the perplexity $\text{Perp} = 2^H$.

**(c)** For a dataset of $N = 10$ molecules, t-SNE symmetrizes the probabilities as $p_{ij} = (p_{j|i} + p_{i|j}) / 2N$. If $p_{j|i} = 0.45$ and $p_{i|j} = 0.25$, compute $p_{ij}$. Why is symmetrization needed?

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
# Scratch: check A2 here
d = np.array([1.0, 1.0, 2.0])

## A3. Heavy tails

**(a)** At what map distance $d$ does the Student-t kernel $(1 + d^2)^{-1}$ fall to 10% of its value at $d = 0$?

**(b)** Answer the same question for the Gaussian kernel $\exp(-d^2)$.

**(c)** In one or two sentences, relate your answers to the crowding problem.

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
# Scratch: check A3 here

## A4. UMAP fuzzy weights

For molecule $i$ with $k = 4$ nearest neighbors at distances 0.5, 1.0, 1.5, and 2.0, UMAP uses

$$w_{ij} = \exp\left(-\frac{d_{ij} - \rho_i}{\sigma_i}\right), \qquad \sum_j w_{ij} = \log_2 k$$

where $\rho_i$ is the distance to the nearest neighbor.

**(a)** Show that the normalization condition reduces to $x + x^2 + x^3 = 1$, where $x = \exp(-0.5/\sigma_i)$.

**(b)** Solve for $x$ numerically (for example by bisection) and find $\sigma_i$.

**(c)** If the weight from $i$ to its second neighbor is $w_{ij}$ and the reverse weight is $w_{ji} = 0.5$, compute the symmetrized weight using the fuzzy union $w = a + b - ab$.

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
# Scratch: check A4 here (e.g. write a small bisection loop)

---
# Part B: Computational practice

The code below builds a dataset of about 90 small molecules in six chemical classes, computes 11 RDKit descriptors, and applies PCA, t-SNE, and UMAP. Run every cell in order, then answer the **YOUR TURN** questions in the answer cells that follow them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors
except ImportError:
    raise ImportError("RDKit is missing. Install it with:  pip install rdkit")

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    print("umap-learn not found; UMAP cells will be skipped. Install with: pip install umap-learn")

SEED = 42

## 1. Build a small molecule dataset
Six chemical classes, generated systematically so every SMILES is valid.

In [ ]:
def build_dataset():
    rows = []

    def add(smiles, cls):
        rows.append({"smiles": smiles, "class": cls})

    for n in range(1, 11):
        add("C" * n + "O", "Alcohol")
    for m in range(1, 6):
        add("CC(O)" + "C" * m, "Alcohol")

    for n in range(1, 11):
        add("C" * n + "N", "Amine")
    for m in range(1, 6):
        add("CN(C)" + "C" * m, "Amine")

    for n in range(0, 10):
        add("C" * n + "C(=O)O", "Carboxylic acid")
    for m in range(1, 6):
        add("OC(=O)" + "C" * m + "C(=O)O", "Carboxylic acid")

    for n in range(1, 11):
        add("CC(=O)" + "C" * n, "Ketone")

    for n in range(1, 9):
        add("C" * n + "Cl", "Haloalkane")
    for n in range(1, 8):
        add("C" * n + "Br", "Haloalkane")

    for n in range(0, 10):
        add("c1ccccc1" + "C" * n, "Aromatic")
    for s in ["Cc1ccc(C)cc1", "Cc1cccc(C)c1", "c1ccc2ccccc2c1",
              "c1ccc(cc1)-c1ccccc1", "c1ccc2cc3ccccc3cc2c1"]:
        add(s, "Aromatic")

    return pd.DataFrame(rows)


def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "MolMR": Crippen.MolMR(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "Rings": rdMolDescriptors.CalcNumRings(mol),
        "AromRings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "FracCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
    }


df = build_dataset()
desc = pd.DataFrame([compute_descriptors(s) for s in df["smiles"]])
data = pd.concat([df, desc], axis=1)
feature_cols = list(desc.columns)
X_raw = data[feature_cols].to_numpy(dtype=float)

print(f"{len(data)} molecules, {len(feature_cols)} descriptors")
print(data["class"].value_counts())
data.head()

## 2. Plotting helper

In [ ]:
CLASSES = sorted(data["class"].unique())
MARKERS = ["o", "^", "s", "D", "v", "P"]
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6"]


def plot_by_class(ax, Y, title, xlabel="Dim 1", ylabel="Dim 2"):
    for cls, mk, col in zip(CLASSES, MARKERS, COLORS):
        idx = (data["class"] == cls).to_numpy()
        ax.scatter(Y[idx, 0], Y[idx, 1], marker=mk, c=col, s=35,
                   edgecolor="white", linewidth=0.5, label=cls)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def plot_by_value(ax, Y, values, title, label):
    sc = ax.scatter(Y[:, 0], Y[:, 1], c=values, cmap="viridis", s=35,
                    edgecolor="white", linewidth=0.5)
    plt.colorbar(sc, ax=ax, label=label)
    ax.set_title(title, fontsize=10)

## 3. PCA: raw vs standardized descriptors

In [ ]:
X_std = StandardScaler().fit_transform(X_raw)

pca_raw = PCA().fit(X_raw - X_raw.mean(axis=0))
pca_std = PCA().fit(X_std)

print("Explained variance ratio (raw):         ", np.round(pca_raw.explained_variance_ratio_[:4], 4))
print("Explained variance ratio (standardized):", np.round(pca_std.explained_variance_ratio_[:4], 4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
Y_raw = pca_raw.transform(X_raw - X_raw.mean(axis=0))[:, :2]
Y_pca = pca_std.transform(X_std)[:, :2]
plot_by_class(axes[0], Y_raw, "PCA on raw descriptors", "PC1", "PC2")
plot_by_class(axes[1], Y_pca, "PCA on standardized descriptors", "PC1", "PC2")

k = np.arange(1, len(feature_cols) + 1)
axes[2].plot(k, np.cumsum(pca_raw.explained_variance_ratio_), "o-", label="raw")
axes[2].plot(k, np.cumsum(pca_std.explained_variance_ratio_), "s--", label="standardized")
axes[2].set_xlabel("Number of components")
axes[2].set_ylabel("Cumulative explained variance")
axes[2].set_title("Scree (cumulative)", fontsize=10)
axes[2].legend()
axes[1].legend(fontsize=8, loc="best")
plt.tight_layout()
plt.savefig("fig1_pca.png", dpi=150)
plt.show()

### PCA loadings (standardized)

In [ ]:
loadings = pd.DataFrame(pca_std.components_[:2].T, index=feature_cols, columns=["PC1", "PC2"])
print(loadings.round(3).sort_values("PC1", key=abs, ascending=False))

### YOUR TURN (B1)
1. Why does raw-data PCA give a very high PC1 explained variance? Which descriptor dominates?
2. Using the loadings, give a chemical name to PC1 and PC2 of the standardized PCA.
3. How many components are needed to reach 90% of the variance (standardized)?

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
n90 = int(np.searchsorted(np.cumsum(pca_std.explained_variance_ratio_), 0.90) + 1)
print("Components needed for 90% variance (standardized):", n90)

## 4. t-SNE: effect of perplexity and random seed

In [ ]:
perplexities = [5, 15, 30, 50]
fig, axes = plt.subplots(1, len(perplexities), figsize=(18, 4.2))
tsne_results = {}
for ax, p in zip(axes, perplexities):
    Y = TSNE(n_components=2, perplexity=p, init="pca", learning_rate="auto",
             random_state=SEED).fit_transform(X_std)
    tsne_results[p] = Y
    plot_by_class(ax, Y, f"t-SNE, perplexity = {p}", "t-SNE 1", "t-SNE 2")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig2_tsne_perplexity.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, seed in zip(axes, [0, 1, 2]):
    Y = TSNE(n_components=2, perplexity=15, init="random", learning_rate="auto",
             random_state=seed).fit_transform(X_std)
    plot_by_class(ax, Y, f"t-SNE, perplexity = 15, seed = {seed}", "t-SNE 1", "t-SNE 2")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig("fig3_tsne_seeds.png", dpi=150)
plt.show()

### YOUR TURN (B2)
1. How does the map change as perplexity increases from 5 to 50?
2. Which features stay the same across seeds, and which change?
3. Why is perplexity = 100 not allowed for this dataset?

**Your answer:**

*(Double-click this cell and write your answer here.)*

## 5. UMAP: effect of n_neighbors and min_dist

In [ ]:
umap_results = {}
if HAVE_UMAP:
    settings = [(5, 0.1), (15, 0.1), (50, 0.1), (15, 0.8)]
    fig, axes = plt.subplots(1, len(settings), figsize=(18, 4.2))
    for ax, (nn, md) in zip(axes, settings):
        Y = umap.UMAP(n_neighbors=nn, min_dist=md, random_state=SEED).fit_transform(X_std)
        umap_results[(nn, md)] = Y
        plot_by_class(ax, Y, f"UMAP, n_neighbors = {nn}, min_dist = {md}", "UMAP 1", "UMAP 2")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig("fig4_umap.png", dpi=150)
    plt.show()

### YOUR TURN (B3)
1. Compare n_neighbors = 5 and 50. Which shows more local detail, which more global layout?
2. What does increasing min_dist from 0.1 to 0.8 do to the clusters?

**Your answer:**

*(Double-click this cell and write your answer here.)*

## 6. Colour by a continuous property

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
plot_by_value(axes[0], Y_pca, data["LogP"], "PCA coloured by LogP", "LogP")
plot_by_value(axes[1], tsne_results[30], data["LogP"], "t-SNE coloured by LogP", "LogP")
if HAVE_UMAP:
    plot_by_value(axes[2], umap_results[(15, 0.1)], data["LogP"], "UMAP coloured by LogP", "LogP")
else:
    axes[2].axis("off")
plt.tight_layout()
plt.savefig("fig5_logp.png", dpi=150)
plt.show()

### YOUR TURN (B4)
Inside each class cluster, is there a smooth LogP gradient? What structural change
(hint: chain length) produces it?

**Your answer:**

*(Double-click this cell and write your answer here.)*

## 7. Quantitative comparison: trustworthiness
Trustworthiness (0 to 1) measures how well each point's nearest neighbours in the map
were also neighbours in the original space. 1 means perfect local preservation.

In [ ]:
emb = {"PCA (2 PCs)": Y_pca, "t-SNE (perp 30)": tsne_results[30]}
if HAVE_UMAP:
    emb["UMAP (nn 15)"] = umap_results[(15, 0.1)]

for k_nn in [5, 15]:
    for name, Y in emb.items():
        t = trustworthiness(X_std, Y, n_neighbors=k_nn)
        print(f"k = {k_nn:2d}  {name:18s} trustworthiness = {t:.3f}")

### YOUR TURN (B5)
1. Which method preserves local neighbourhoods best? Is that what you expected?
2. Trustworthiness says nothing about distances between clusters. Propose a check
   for global structure (hint: correlate pairwise distances in the original and 2D spaces).

**Your answer:**

*(Double-click this cell and write your answer here.)*

In [ ]:
# Your code for the global-structure check
# Hint: from scipy.spatial.distance import pdist; from scipy.stats import spearmanr

---
# Part C: Short report

In no more than one page (about 400 words), answer the following question:

> You are given 5,000 DFT-computed single-atom catalysts, each described by 50 descriptors. You want to (i) find clusters of chemically similar catalysts and (ii) identify which descriptors drive the differences between them. Which method(s) would you use for each goal, in what order, and what pitfalls would you watch for?

Support your answer with at least one observation from your own results in Part B.

**Your answer:**

*(Double-click this cell and write your answer here.)*

---
# Bonus (optional)

Repeat the t-SNE and UMAP analysis using 2048-bit Morgan fingerprints (radius 2) instead of descriptors, with the Jaccard (Tanimoto) metric in UMAP. Do the clusters change? Explain why.

Hint: `from rdkit.Chem import rdFingerprintGenerator` and `rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)`.

In [ ]:
# Your bonus code here

**Your answer:**

*(Double-click this cell and write your answer here.)*